# RAG 07: PDF Parsing and Cleaning

This notebook starts the offline RAG pipeline.

The goal is to turn the OWASP PDF into inspectable LangChain `Document` objects. Each page becomes one document with text in `page_content` and source information in `metadata`.

In [ ]:
from pathlib import Path
import json
import os
import re

from dotenv import load_dotenv
from langchain_core.documents import Document
from pypdf import PdfReader

load_dotenv()

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent


def repo_path(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return REPO_ROOT / path


PDF_PATH = repo_path(os.getenv("OWASP_LLM_PDF_PATH", "data/owasp_top10_llm_applications.pdf"))
PAGES_PATH = repo_path(os.getenv("OWASP_LLM_PAGES_PATH", "data/owasp_top10_llm_pages.jsonl"))

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PDF_PATH}. Add the OWASP PDF to data/ before running this notebook."
    )

## Extract Pages

PDF extraction is not the same as reading a clean text file. A PDF is a layout format, so extraction can produce broken line breaks, repeated spaces, and empty pages.

In [ ]:
def clean_page_text(text: str) -> str:
    # Keep headings and sentences, but remove layout noise that hurts retrieval.
    text = text.replace("\x00", " ")
    text = re.sub(r"[ 	]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

reader = PdfReader(str(PDF_PATH))
pdf_title = ""
if reader.metadata and reader.metadata.title:
    pdf_title = str(reader.metadata.title)

page_documents = []

for page_index, page in enumerate(reader.pages, start=1):
    raw_text = page.extract_text() or ""
    cleaned_text = clean_page_text(raw_text)

    if not cleaned_text:
        continue

    page_documents.append(
        Document(
            page_content=cleaned_text,
            metadata={
                "source": str(PDF_PATH),
                "title": pdf_title or "OWASP Top 10 for LLM Applications",
                "page": page_index,
                "kind": "pdf_page",
            },
        )
    )

print({
    "pdf_path": str(PDF_PATH),
    "pdf_pages": len(reader.pages),
    "non_empty_pages": len(page_documents),
    "title": pdf_title,
})

## Inspect One Page Document

A `Document` separates the searchable text from metadata. Metadata is how the final answer can cite the page it used.

In [ ]:
sample = page_documents[min(2, len(page_documents) - 1)]

print(sample.metadata)
print(sample.page_content[:1000])

## Save Parsed Pages

The next notebook starts from this JSONL file instead of parsing the PDF again.

In [ ]:
PAGES_PATH.parent.mkdir(parents=True, exist_ok=True)

with PAGES_PATH.open("w", encoding="utf-8") as file:
    for document in page_documents:
        file.write(
            json.dumps(
                {
                    "content": document.page_content,
                    "metadata": document.metadata,
                },
                ensure_ascii=False,
            )
            + "\n"
        )

print({"pages_path": str(PAGES_PATH), "documents_saved": len(page_documents)})